Język: [English](lab.en.ipynb) | **Polski**

# Laboratorium 11 — projektowanie i ocena niewielkiego pipeline'u inferencji LLM

Materiał towarzyszący [`README.pl.md`](README.pl.md). Pipeline: prompt → tokenizacja →
model → generowanie → dekodowanie → pomiar → ocena.

**Rdzeń obowiązkowy: sekcje 0-11 (ok. 105 min + 15 min bufora = 120 min).**
Sekcje R1-R2 na końcu notebooka to **rozszerzenia opcjonalne**.

In [ ]:
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)
torch.set_num_threads(4)
device = torch.device("cpu")
print(f"Uzywane urzadzenie: {device}")

MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
REVISION = "12fd25f77366fa6b3b4b768ec3050bf629380bac"

## 1. Model i tokenizer (gotowe + krótkie zadanie)

In [ ]:
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, dtype=torch.float32)
model.eval()
print(f"load time: {time.time() - t0:.2f}s")
print(f"liczba parametrow: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# TODO: wypisz model.config.num_attention_heads i model.config.num_key_value_heads
# Czy sa rowne? Co to oznacza dla kosztu pamieci KV cache? (zobacz wyklad)
raise NotImplementedError("TODO: sprawdzenie glow Q vs KV")

## 2. Przygotowanie promptów — Twoje zadanie

Wybierz **dokładnie 3 obowiązkowe** prompty, pokrywające:
1. **jeden** prompt jednoznaczny typu "ekstrakcja informacji" (ma precyzyjną
   oczekiwaną odpowiedź),
2. **jeden drugi** prompt jednoznaczny, innego typu (np. "przekształcenie
   formatu"),
3. **jeden** prompt **jakościowy** (parafraza albo kontynuacja — `expected=None`).

Dla **każdego** promptu z jednoznaczną odpowiedzią, zapisz tę odpowiedź **z
góry**, zanim zobaczysz, co model wygeneruje. Struktura: lista słowników
`{"id", "typ", "prompt", "expected"}`.

**Rozszerzenie R1 (opcjonalne):** dodaj 2 kolejne prompty (np. pytanie z
kontekstem, kontynuacja) z pełnym traktowaniem jak w rdzeniu.

In [ ]:
# TODO: zbuduj liste DOKLADNIE 3 obowiazkowych slownikow {"id","typ","prompt","expected"}:
# 1. jeden prompt jednoznaczny (np. ekstrakcja informacji), expected != None
# 2. jeden DRUGI prompt jednoznaczny, innego typu (np. przeksztalcenie formatu)
# 3. jeden prompt JAKOSCIOWY (parafraza albo kontynuacja), expected=None
prompts = []
raise NotImplementedError("TODO: przygotowanie promptow")

## 3. Chat template — Twoje zadanie

Zaimplementuj `format_prompt(prompt_text)`, która zwraca `inputs` gotowe do
`model.generate(**inputs, ...)`, używając `tokenizer.apply_chat_template()`
z `add_generation_prompt=True`. **Nie** dodawaj ręcznie żadnych tokenów
specjalnych — `apply_chat_template` już je wstawia (zobacz wykład, sekcja
"Chat templates").

**Uwaga API (zweryfikowana w wykładzie):** `apply_chat_template(...,
tokenize=True, return_tensors="pt")` **bez** `return_dict=True` może zwrócić
inny typ niż oczekujesz w Twojej wersji `transformers` — użyj
`return_dict=True` jawnie i odwołuj się przez `inputs["input_ids"]`, nie
zakładaj gołego tensora.

**Zadanie (kod):** zweryfikuj, że Twoja funkcja faktycznie dodaje tokeny
specjalne (sprawdź, że zdekodowany prompt zaczyna się od tokenu BOS/
`<|im_start|>`, nie od treści promptu).

In [ ]:
def format_prompt(prompt_text):
    # TODO:
    # 1. zbuduj messages = [{"role": "user", "content": prompt_text}]
    # 2. zwroc tokenizer.apply_chat_template(messages, tokenize=True,
    #    add_generation_prompt=True, return_tensors="pt", return_dict=True)
    raise NotImplementedError("TODO: format_prompt")

**Weryfikacja (gotowa asercja): zdekodowany początek sformatowanego promptu
musi zaczynać się od tokenu specjalnego (BOS/`<|im_start|>`), nie od treści.**

In [ ]:
check_inputs = format_prompt("test")  # noqa: F821 - zdefiniowane w sekcji 3
start_decoded = tokenizer.decode(check_inputs["input_ids"][0][:1])
print(f"pierwszy token sformatowanego promptu: {start_decoded!r}")
assert start_decoded == tokenizer.bos_token, (
    f"BLAD: oczekiwano {tokenizer.bos_token!r} jako pierwszy token, dostano {start_decoded!r}"
)
print("OK: format_prompt dodaje token specjalny na starcie.")

## 4. Generowanie i poprawne dekodowanie — Twoje zadanie

Zaimplementuj `generate_response(prompt_text, **gen_kwargs)`:
1. sformatuj prompt (sekcja 3),
2. wywołaj `model.generate(**inputs, **gen_kwargs, pad_token_id=...)`,
3. **wyciągnij TYLKO nowo wygenerowane tokeny** (nie cały `output`, który
   zawiera też prompt — zobacz wykład, Eksperyment B/E),
4. zdekoduj (`skip_special_tokens=True`),
5. zwróć tekst **oraz** liczbę nowo wygenerowanych tokenów (potrzebne do
   pomiaru przepustowości w sekcji 6).

In [ ]:
def generate_response(prompt_text, **gen_kwargs):
    # TODO:
    # 1. inputs = format_prompt(prompt_text)
    # 2. prompt_len = inputs["input_ids"].shape[1]
    # 3. out = model.generate(**inputs, pad_token_id=tokenizer.pad_token_id, **gen_kwargs)
    # 4. WYCIAGNIJ TYLKO nowo wygenerowane tokeny: out[0, prompt_len:] (NIE caly out!)
    # 5. zdekoduj (skip_special_tokens=True)
    # 6. zwroc (text, liczba_nowych_tokenow)
    raise NotImplementedError("TODO: generate_response")

**Weryfikacja (gotowa asercja): dla `do_sample=False`, dwa wywołania z tym
samym promptem muszą dać identyczny wynik.**

In [ ]:
r1, _ = generate_response("What is the capital of France?", max_new_tokens=10, do_sample=False)  # noqa: F821
r2, _ = generate_response("What is the capital of France?", max_new_tokens=10, do_sample=False)  # noqa: F821
print(f"przebieg 1: {r1!r}")
print(f"przebieg 2: {r2!r}")
assert r1 == r2, "BLAD: greedy decoding powinien byc deterministyczny"
print("OK: deterministyczne powtorzenia sa identyczne.")

## 5. Deterministyczne generowanie i sampling — Twoje zadanie

Dla **wszystkich 3 obowiązkowych** promptów (sekcja 2), wygeneruj odpowiedź
**greedy** (`do_sample=False`). Dodatkowo, dla **podzbioru** (prompt
jednoznaczny `extract` **i** prompt jakościowy `paraphrase` — **nie**
`format`, żeby nie wymagać pełnego iloczynu 3 prompty × 3 strategie), uruchom
też **sampling** z dwiema temperaturami (0,7 i 1,3). Użyj **ustalonego
ziarna** (`torch.manual_seed(...)`) przed każdym wywołaniem samplingu.

**Rozszerzenie R2 (opcjonalne):** uruchom sampling również dla `format` (i
dla promptów z R1, jeśli je dodałaś/eś) — czy sampling pomaga na zadaniach
jednoznacznych, którym rdzeń go nie daje?

In [ ]:
configs_greedy = [("greedy", dict(do_sample=False))]
configs_sampling = [
    ("sampling tau=0.7", dict(do_sample=True, temperature=0.7)),
    ("sampling tau=1.3", dict(do_sample=True, temperature=1.3)),
]
SAMPLING_SUBSET = {"extract", "paraphrase"}  # id-i promptow, ktore dodatkowo dostaja sampling

# TODO:
# dla kazdego promptu (sekcja 2): zawsze greedy; DODATKOWO sampling (oba warianty
# z configs_sampling) TYLKO jesli p["id"] in SAMPLING_SUBSET
# 1. ustaw torch.manual_seed(0) PRZED kazdym wywolaniem (powtarzalnosc)
# 2. zmierz czas (time.time() przed/po) wywolania
#    generate_response(prompt, max_new_tokens=40, **kwargs)
# 3. zbierz wyniki (lista slownikow: id, strategy, text, n_new, time, tok_s)
results = []
raise NotImplementedError("TODO: generowanie - greedy dla wszystkich, sampling dla podzbioru")

## 6. Pomiar czasu i przepustowości (gotowe — już zebrane w sekcji 5)

**Zadanie:** wypisz tabelę tokenów/sekundę dla każdej kombinacji (prompt,
strategia) z `results`.

In [ ]:
# TODO: wypisz id, strategie i tok/s dla kazdego wpisu w `results`
raise NotImplementedError("TODO: tabela przepustowosci")

## 7. Wpływ długości promptu — Twoje zadanie

Wybierz **jeden** prompt i wygeneruj jego dłuższą wersję (np. dodaj
kontekst/historię). Zmierz czas generowania **tej samej liczby** nowych
tokenów dla krótkiej i długiej wersji promptu.

**Ważne:** użyj `min_new_tokens == max_new_tokens`, żeby wymusić **tę samą**
liczbę nowych tokenów dla obu długości promptu — inaczej różnice w czasie
generowania (np. wcześniejsze zatrzymanie przez EOS) zanieczyszczą pomiar
samego wpływu długości promptu (ta pułapka jest realna — sprawdzona przy
przygotowaniu tego laboratorium).

In [ ]:
short_prompt = "What is 2+2?"
long_prompt = ("Here is a lot of context to pad out the prompt. " * 30) + " What is 2+2?"

# TODO: dla obu promptow, zmierz czas generate_response z max_new_tokens=20 I
# min_new_tokens=20 (ta sama liczba nowych tokenow dla obu!), wypisz prompt_len i czas
raise NotImplementedError("TODO: wplyw dlugosci promptu")

## 8. KV cache — Twoje zadanie

In [ ]:
# TODO: dla use_cache=True i use_cache=False, zmierz czas generate_response
# ("Tell me about the solar system.", max_new_tokens=40, do_sample=False, use_cache=...)
raise NotImplementedError("TODO: KV cache")

## 9. Ocena jakości — Twoje zadanie

1. **Automatycznie**, dla **obu** promptów z jednoznaczną odpowiedzią
   (sekcja 2) — na ich odpowiedzi **greedy** — sprawdź
   `p["expected"].lower() in response.lower()`.
2. **Jakościowo**, dla promptu **jakościowego** (sekcja 2) — oceń **każdą**
   uruchomioną strategię (greedy + 2 warianty samplingu z sekcji 5) wg
   pięciu kryteriów z README (poprawność wykonania, kompletność, zgodność z
   formatem, naturalność, powtarzalność) — z **konkretnym uzasadnieniem
   opartym na fragmencie odpowiedzi**, nie "wygląda dobrze".

❌ "Ta odpowiedź wygląda lepiej, więc jest lepsza."

✅ Rozdzielamy te kryteria właśnie, żeby **nie** oceniać jednym, subiektywnym
wrażeniem.

**Rozszerzenie R1/R2 (opcjonalne):** jeśli wykonałaś/eś R1 (dodatkowe
prompty) lub R2 (dodatkowy sampling), oceń je tym samym schematem.

In [ ]:
# TODO: dla obu promptow z expected != None, sprawdz czy
# p["expected"].lower() in <odpowiedz greedy tego promptu>.lower()
# wypisz id, expected, wynik sprawdzenia, i sama odpowiedz
raise NotImplementedError("TODO: automatyczna ocena")

**Zadanie (markdown):** dla promptu `paraphrase` (jedyny jakościowy prompt w
rdzeniu), oceń **każdą** z uruchomionych strategii (greedy, tau=0,7, tau=1,3)
pod względem: czy zachowuje znaczenie oryginału, czy jest kompletna, czy
brzmi naturalnie. Konkretne uzasadnienie, nie "wygląda dobrze".

_(Twoja odpowiedź tutaj.)_

## 10. Zadanie diagnostyczne

Poniższy kod **wygląda** poprawnie — nie rzuca wyjątku, generuje tekst — ale
ma błąd: pomylenie `max_length` z `max_new_tokens`.

In [ ]:
DESIRED_NEW_TOKENS = 40
diag_prompt = "Write a short poem about the ocean."
diag_inputs = format_prompt(diag_prompt)  # noqa: F821
diag_prompt_len = diag_inputs["input_ids"].shape[1]
print(f"prompt_len={diag_prompt_len}, chcemy {DESIRED_NEW_TOKENS} NOWYCH tokenow")

with torch.no_grad():
    out_buggy = model.generate(
        **diag_inputs, max_length=DESIRED_NEW_TOKENS, do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
actual_new_buggy = out_buggy.shape[1] - diag_prompt_len
print(f"BuggyGeneration: faktycznie nowych tokenow = {actual_new_buggy} "
      f"(chcielismy {DESIRED_NEW_TOKENS}!)")
text_buggy = tokenizer.decode(out_buggy[0, diag_prompt_len:], skip_special_tokens=True)
print(f"tekst (uciety): {text_buggy!r}")

**Zadanie:**
1. Uruchom kod z krótkim promptem i zaobserwuj: ile **nowych** tokenów
   faktycznie powstało, względem tego, ile **chciał** autor kodu (zmienna
   `DESIRED_NEW_TOKENS`)?
2. Zdiagnozuj przyczynę: co dokładnie liczy `max_length`, a co
   `max_new_tokens`? (Zobacz wykład, sekcja "Okno kontekstowe i
   obcinanie".)
3. Uruchom **ten sam** kod z **dłuższym** promptem (wystarczająco długim,
   żeby `prompt_len >= max_length`) — co się teraz dzieje? (Podpowiedź:
   inne, głośniejsze zachowanie niż przy krótkim prompcie.)
4. Popraw kod (`max_new_tokens=` zamiast `max_length=`).
5. Zweryfikuj poprawkę: dla **obu** długości promptu, liczba nowo
   wygenerowanych tokenów musi być zgodna z zamierzoną.

In [ ]:
# TODO:
# 1. jako komentarz: zdiagnozuj dokladnie, co max_length liczy tutaj, a co chcielismy
# 2. sprobuj TEJ SAMEJ buggy konfiguracji z DLUZSZYM promptem (prompt_len >= max_length)
#    - co sie teraz dzieje? (podpowiedz: inny, glosniejszy blad)
# 3. napraw: uzyj max_new_tokens= zamiast max_length=
# 4. zweryfikuj POPRAWKE dla OBU dlugosci promptu (assert na liczbie nowych tokenow)
raise NotImplementedError("TODO: diagnoza i poprawka max_length/max_new_tokens")

## 11. Wnioski — Twoje zadanie

**1. Która strategia dekodowania dała najbardziej użyteczne wyniki dla zadań z jednoznaczną odpowiedzią (ekstrakcja, format)? Czy sampling (tam, gdzie go przetestowałaś/eś) pomógł, zaszkodził, czy nie miał znaczenia?**

_(Twoja odpowiedź tutaj.)_

**2. Dla zadania jakościowego — czy wyższa temperatura dała subiektywnie lepsze wyniki? Czy różnorodność miała znaczenie dla tego konkretnego zadania?**

_(Twoja odpowiedź tutaj.)_

**3. Jaki jest zmierzony koszt pipeline'u? Jak zmienia się z długością promptu i z `use_cache`?**

_(Twoja odpowiedź tutaj.)_

**4. Jakie konkretne ograniczenia modelu/pipeline'u zaobserwowałaś/eś?**

_(Twoja odpowiedź tutaj.)_

**5. Jaki jest Twój wniosek o kompromisie jakość/koszt/konfiguracja — czy 'lepsza' konfiguracja zawsze oznacza lepszy wynik dla każdego typu zadania?**

_(Twoja odpowiedź tutaj.)_

---

## Rozszerzenia (opcjonalne, jeśli zostanie czas po sekcjach 0-11)

- **R1.** Dodaj 2 kolejne prompty (np. pytanie z kontekstem, kontynuacja
  tekstu) do listy z sekcji 2 i przeprowadź dla nich to samo traktowanie
  (sekcje 5, 6, 9) co dla promptów obowiązkowych.
- **R2.** Uruchom sampling (obie temperatury) również dla `format` (i dla
  promptów z R1, jeśli je dodałaś/eś) — czy sampling pomaga, szkodzi, czy
  nie ma znaczenia dla zadań jednoznacznych, którym rdzeń go nie dał?
- Dodaj **szósty** prompt własnego projektu, w innym stylu niż propozycje
  wyżej.
- Zmierz wpływ `repetition_penalty` na kontynuację tekstu (czy zmniejsza
  powtarzalność fraz przy dłuższym generowaniu?).

_(To rozszerzenie jest opcjonalne — kod i wyniki nie są wymagane w rdzeniu.)_